In [1]:
from pathlib import Path
import pandas as pd
from natsort import natsorted

In [2]:
def announce(s):
    print('-'*70)
    print(s)
    print('-'*70)

def load_trajectory_pickle(traj_path) -> pd.DataFrame:
    if not traj_path.exists():
        raise FileNotFoundError(f"Trajectory file not found: {traj_path}")
    return pd.read_pickle(traj_path, compression="gzip")

def get_samples_last_point(traj_df: pd.DataFrame) -> pd.DataFrame:
    last_points = (
        traj_df.sort_values(["sample", "time"])
        .groupby("sample", as_index=False)
        .tail(1)
        .copy()
    )
    return last_points

def classify_major_product_last_point(
    last_points: pd.DataFrame,
    product_a: str = "map",
    product_b: str = "bis",
) -> pd.DataFrame:
    required_cols = {"sample", "time", product_a, product_b}
    missing = required_cols - set(last_points.columns)
    if missing:
        raise KeyError(f"Missing required columns in trajectory data: {sorted(missing)}")

    last_points["major_product"] = last_points.apply(
        lambda r: product_a if r[product_a] > r[product_b]
        else (product_b if r[product_b] > r[product_a] else "tie"),
        axis=1,
    )

    return last_points["major_product"]

In [3]:
# Folder where mode2 trajectory pickle files are expected
TRAJ_DIR = Path("exports_persist/validation_dft-opt_B2")

# List the contents of TRAJ_DIR and select all .pkl.gz files
if not TRAJ_DIR.exists():
    raise FileNotFoundError(f"Trajectory directory not found: {TRAJ_DIR}")

announce(f"Listing trajectory files in {TRAJ_DIR}")
traj_files = natsorted(TRAJ_DIR.glob("*.pkl.gz"))
if not traj_files:
    raise FileNotFoundError(f"No trajectory pickle files found in {TRAJ_DIR}. Run Mode 2 export first.")
for i, traj_file in enumerate(traj_files, 1):
    print(f"{i}. {traj_file.name}")

# Create a alias for the trajectory file selection (dictionary mapping alias to file path)
# The alias is _filename.split(_, 1)[0]
traj_aliases = {f.stem.split("_", 1)[0]: f for f in traj_files}
announce("Available trajectory file aliases:")
for alias in traj_aliases:
    print(f"- {alias}")

# Now we create the mapping
picle_mapping = {alias: traj_aliases[alias] for alias in traj_aliases}
announce("Trajectory file mapping:")
for alias, path in picle_mapping.items():
    print(f"{alias}: {path}")

----------------------------------------------------------------------
Listing trajectory files in exports_persist/validation_dft-opt_B2
----------------------------------------------------------------------
1. im1-ad1_mode2_trajectories.pkl.gz
2. im1-oc1_mode2_trajectories.pkl.gz
3. im1-oc3_mode2_trajectories.pkl.gz
4. im2-bitet0_mode2_trajectories.pkl.gz
5. im2-oc0_mode2_trajectories.pkl.gz
6. im2-oc1_mode2_trajectories.pkl.gz
7. im2-oc3_mode2_trajectories.pkl.gz
8. im3-15_mode2_trajectories.pkl.gz
9. im3-26_mode2_trajectories.pkl.gz
10. im3-29_mode2_trajectories.pkl.gz
11. im3-31_mode2_trajectories.pkl.gz
12. im4-oc1_mode2_trajectories.pkl.gz
13. im5-oc1_mode2_trajectories.pkl.gz
14. im5-oc2_mode2_trajectories.pkl.gz
15. im5-oc3_mode2_trajectories.pkl.gz
16. im5-oc4_mode2_trajectories.pkl.gz
----------------------------------------------------------------------
Available trajectory file aliases:
----------------------------------------------------------------------
- im1-ad1
- im1-o

In [4]:
CONVERSION_THRESHOLD_PERCENT = 5

def mode2_pipeline_parser(pickle_path:str, verbose:bool=True) -> pd.DataFrame:

    #####################################################
    # LOAD PICKLE DATA                                  #
    #####################################################

    # Load the trajectory data (pickle file)
    traj_df = load_trajectory_pickle(pickle_path)
    if verbose:
        print(f"\nLoaded trajectory data from {pickle_path} with shape {traj_df.shape}\n")
        announce("Parsing the trajectory data...")

    # Drop the column me2pyr if it exists, as it's not relevant for the analysis
    if "me2pyr" in traj_df.columns:
        traj_df.drop(columns=["me2pyr"], inplace=True)
        if verbose:
            print("    🐸 Dropped 'me2pyr' column from trajectory data")
    
    # The expected trajectory data have the % of each species instead of the concentrations
    # Because of this, we multiply every column after "time" by 100
    _time_col_idx = traj_df.columns.get_loc("time")
    traj_df.iloc[:, _time_col_idx + 1:] *= 100
    if verbose:
        print("    🐸 Calculated percentages for species columns by multiplying by 100")

    # Calculate the conversion by calculating 100 - bispyr
    traj_df["conversion"] = 100 - traj_df["bispyr"]

    if verbose:
        print("    🐸 Calculated conversion for each trajectory point")

    if verbose:
        print("Trajectory data after processing:")
        display(traj_df.head())
        display(traj_df.info())

    #####################################################
    # LAST POINT DATA                                   #
    #####################################################

    if verbose:
        announce("Extracting last points for each sample and classifying major product...")

    # Get the last point of each sample
    last_points = get_samples_last_point(traj_df)
   
    # Specify the major product for each sample based on the last point
    last_points['major_product'] = classify_major_product_last_point(last_points)
    if verbose:
        print("    🐸 Classified major product for each sample based on the last point")

    # Calculate the f_map and f_bis for each row, where f_map = map / (map + bis) and f_bis = bis / (map + bis)
    _products = (last_points["map"] + last_points["bis"])
    last_points["f_map"] = last_points["map"] / _products
    last_points["f_bis"] = last_points["bispyr"] / _products
    if verbose:
        print("    🐸 Calculated f_map and f_bis for each sample")

    if verbose:
        print("Last points data after classification and feature engineering:")
        display(last_points.head())
        display(last_points.info())


    #####################################################
    # AGG METRICS - NO BASELINE                         #
    #####################################################

    # Now we start the calculation of aggregate metrics
    _agg_data_df = last_points.copy()
    
    # Drop rows where is_baseline == 1 because this is the unperturbated (non MC) trajectory
    _agg_data_df = _agg_data_df[_agg_data_df["is_baseline"] == 0].copy()
    # Drop the is_baseline column as it's not needed anymore
    _agg_data_df.drop(columns=["is_baseline"], inplace=True)
    # Drop the time column as it's not needed for the aggregate metrics
    _agg_data_df.drop(columns=["time"], inplace=True)
    if verbose:
        announce("Preparing aggregate metrics data...")
        print(f"    🐸 Dropped baseline samples: shape is {_agg_data_df.shape}")
        print(f"    🐸 Dropped 'is_baseline' column from last points data: shape is {_agg_data_df.shape}")
        print(f"    🐸 Dropped 'time' column from last points data: shape is {_agg_data_df.shape}")

    # # Drop samples where the conversion is below the threshold, as these are not considered valid trajectories
    # _agg_data_df = _agg_data_df[_agg_data_df["conversion"] >= CONVERSION_THRESHOLD_PERCENT].copy()
    # if verbose:
    #     print(f"    🐸 Dropped samples with conversion below {CONVERSION_THRESHOLD_PERCENT}%: shape is {_agg_data_df.shape}")

    if verbose:
        print("Data for aggregate metrics calculation:")
        display(_agg_data_df.head())
        display(_agg_data_df.info())

    if verbose:
        announce("Calculating aggregate metrics...")

    # Calculate the map, bis and tie counts for the major product
    map_counts = (_agg_data_df["major_product"] == "map").sum()
    bis_counts = (_agg_data_df["major_product"] == "bis").sum()
    # tie_counts = (_agg_data_df["major_product"] == "tie").sum()

    p_map = map_counts / (map_counts + bis_counts)
    p_bis = bis_counts / (map_counts + bis_counts)
    # p_tie = tie_counts / len(_agg_data_df)

    agg_data = {
        'conversion_avg': _agg_data_df["conversion"].mean(),
        'conversion_std': _agg_data_df["conversion"].std(),
        'conversion_median': _agg_data_df["conversion"].median(),
        'conversion_CI_low': _agg_data_df["conversion"].quantile(0.025),
        'conversion_CI_up': _agg_data_df["conversion"].quantile(0.975),

        'roh_avg': _agg_data_df["roh"].mean(),
        'roh_std': _agg_data_df["roh"].std(),
        'roh_median': _agg_data_df["roh"].median(),
        'roh_CI_low': _agg_data_df["roh"].quantile(0.025),
        'roh_CI_up': _agg_data_df["roh"].quantile(0.975),

        'bispyr_avg': _agg_data_df["bispyr"].mean(),
        'bispyr_std': _agg_data_df["bispyr"].std(),
        'bispyr_median': _agg_data_df["bispyr"].median(),
        'bispyr_CI_low': _agg_data_df["bispyr"].quantile(0.025),
        'bispyr_CI_up': _agg_data_df["bispyr"].quantile(0.975),

        'map_avg': _agg_data_df["map"].mean(),
        'map_std': _agg_data_df["map"].std(),
        'map_median': _agg_data_df["map"].median(),
        'map_CI_low': _agg_data_df["map"].quantile(0.025),
        'map_CI_up': _agg_data_df["map"].quantile(0.975),

        'bis_avg': _agg_data_df["bis"].mean(),
        'bis_std': _agg_data_df["bis"].std(),
        'bis_median': _agg_data_df["bis"].median(),
        'bis_CI_low': _agg_data_df["bis"].quantile(0.025),
        'bis_CI_up': _agg_data_df["bis"].quantile(0.975),

        'f_map_avg': _agg_data_df["f_map"].mean(),
        'f_map_std': _agg_data_df["f_map"].std(),
        'f_map_median': _agg_data_df["f_map"].median(),
        'f_map_CI_low': _agg_data_df["f_map"].quantile(0.025),
        'f_map_CI_up': _agg_data_df["f_map"].quantile(0.975),
        
        'counts_map_major_product': map_counts,
        'counts_bis_major_product': bis_counts,
        'p_map': p_map,
        'p_bis': p_bis,
    }
    agg_data_se = pd.Series(agg_data) 

    if verbose:
        print("Aggregate metrics calculated:")
        display(agg_data_se.round(2))

    return traj_df, last_points, agg_data_se


if False:
    _sample_path = picle_mapping.get("im2-oc0")
    kinetics, last_points, agg_data_se = mode2_pipeline_parser(_sample_path, verbose=True)

In [5]:
agg_data_all = {}

announce("Processing all trajectories to calculate aggregate metrics...")

for alias in traj_aliases:
    print(f"Parsing data for {alias}...")

    kinetics, last_points, agg_data_se = mode2_pipeline_parser(traj_aliases[alias], verbose=False)
    agg_data_all[alias] = agg_data_se.to_dict()

agg_data_df = pd.DataFrame(agg_data_all).T
agg_data_df.index.name = "trajectory_alias"
agg_data_df.reset_index(inplace=True)

----------------------------------------------------------------------
Processing all trajectories to calculate aggregate metrics...
----------------------------------------------------------------------
Parsing data for im1-ad1...
Parsing data for im1-oc1...
Parsing data for im1-oc3...
Parsing data for im2-bitet0...
Parsing data for im2-oc0...
Parsing data for im2-oc1...
Parsing data for im2-oc3...
Parsing data for im3-15...
Parsing data for im3-26...
Parsing data for im3-29...
Parsing data for im3-31...
Parsing data for im4-oc1...
Parsing data for im5-oc1...
Parsing data for im5-oc2...
Parsing data for im5-oc3...
Parsing data for im5-oc4...


In [6]:
announce("Aggregate metrics for all trajectories:")
display(agg_data_df.round(2))
display(agg_data_df.info())

----------------------------------------------------------------------
Aggregate metrics for all trajectories:
----------------------------------------------------------------------


,trajectory_alias,conversion_avg,conversion_std,conversion_median,conversion_CI_low,conversion_CI_up,roh_avg,roh_std,roh_median,roh_CI_low,...,bis_CI_up,f_map_avg,f_map_std,f_map_median,f_map_CI_low,f_map_CI_up,counts_map_major_product,counts_bis_major_product,p_map,p_bis
0,im1-ad1,100.00,0.00,100.00,100.00,100.0,0.01,0.01,0.01,0.01,...,99.99,0.00,0.00,0.00,0.00,0.00,0.0,1000.0,0.00,1.00
1,im1-oc1,100.00,0.00,100.00,100.00,100.0,0.15,0.63,0.09,0.09,...,99.91,0.00,0.01,0.00,0.00,0.01,0.0,1000.0,0.00,1.00
2,im1-oc3,100.00,0.00,100.00,100.00,100.0,81.68,20.37,88.30,15.67,...,84.33,0.82,0.20,0.88,0.16,1.00,908.0,92.0,0.91,0.09
3,im2-bitet0,100.00,0.00,100.00,100.00,100.0,0.10,0.00,0.10,0.10,...,99.90,0.00,0.00,0.00,0.00,0.00,0.0,1000.0,0.00,1.00
4,im2-oc0,100.00,0.00,100.00,100.00,100.0,0.07,0.29,0.04,0.04,...,99.96,0.00,0.00,0.00,0.00,0.00,0.0,1000.0,0.00,1.00
5,im2-oc1,99.98,0.22,100.00,99.93,100.0,0.10,0.44,0.07,0.07,...,99.93,0.00,0.00,0.00,0.00,0.00,0.0,1000.0,0.00,1.00
6,im2-oc3,100.00,0.00,100.00,100.00,100.0,98.07,6.16,99.50,83.15,...,16.85,0.98,0.06,1.00,0.83,1.00,995.0,5.0,1.00,0.00
7,im3-15,99.80,1.83,99.99,99.10,100.0,0.59,3.84,0.09,0.04,...,99.97,0.00,0.01,0.00,0.00,0.01,0.0,1000.0,0.00,1.00
8,im3-26,94.88,15.39,100.00,40.64,100.0,105.12,15.39,100.00,100.00,...,0.00,1.00,0.00,1.00,1.00,1.00,1000.0,0.0,1.00,0.00
9,im3-29,99.46,3.92,100.00,96.28,100.0,10.36,16.50,4.13,0.66,...,99.47,0.09,0.15,0.04,0.00,0.62,41.0,959.0,0.04,0.96


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   trajectory_alias          16 non-null     object 
 1   conversion_avg            16 non-null     float64
 2   conversion_std            16 non-null     float64
 3   conversion_median         16 non-null     float64
 4   conversion_CI_low         16 non-null     float64
 5   conversion_CI_up          16 non-null     float64
 6   roh_avg                   16 non-null     float64
 7   roh_std                   16 non-null     float64
 8   roh_median                16 non-null     float64
 9   roh_CI_low                16 non-null     float64
 10  roh_CI_up                 16 non-null     float64
 11  bispyr_avg                16 non-null     float64
 12  bispyr_std                16 non-null     float64
 13  bispyr_median             16 non-null     float64
 14  bispyr_CI_lo

None

In [7]:
announce("Exporting aggregate metrics to CSV...")
agg_data_df.to_csv(f"{TRAJ_DIR}/aggregate_metrics.csv", index=False)
print(f"Aggregate metrics exported to {TRAJ_DIR}/aggregate_metrics.csv")

----------------------------------------------------------------------
Exporting aggregate metrics to CSV...
----------------------------------------------------------------------
Aggregate metrics exported to exports_persist/validation_dft-opt_B2/aggregate_metrics.csv
